# Notebook 04 — Failure Prediction
**Project:** Predictive Maintenance System  
**Author:** Abrar Ahmed

Models: Logistic Regression (baseline) → XGBoost + SHAP → 1D-CNN  
All experiments tracked in MLflow.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
    f1_score, roc_auc_score, precision_recall_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import mlflow, mlflow.sklearn, mlflow.xgboost, mlflow.pytorch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DATA_PROCESSED = Path('../data/processed')
MODELS_DIR     = Path('../models')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

plt.rcParams.update({'figure.dpi':120,'figure.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3})

print(f'XGBoost : {xgb.__version__}')
print(f'SHAP    : {shap.__version__}')
print(f'Device  : {device}')
print('Imports OK')

In [ ]:
# Load data
df = pd.read_parquet(DATA_PROCESSED / 'features.parquet')
df = df.sort_values(['machineID','datetime']).reset_index(drop=True)

# Feature columns
raw_sensors   = ['volt','rotate','pressure','vibration']
rolling_means = [c for c in df.columns if '_mean_' in c]
rolling_stds  = [c for c in df.columns if '_std_'  in c]
rolling_mins  = [c for c in df.columns if '_min_'  in c]
rolling_maxs  = [c for c in df.columns if '_max_'  in c]
error_feats   = [c for c in df.columns if 'error_' in c and '_24h' in c]
time_feats    = ['hour_of_day','day_of_week','day_of_month','month','is_weekend']
maint_feats   = ['hours_since_maint']
meta_feats    = ['age']

FEATURE_COLS = (raw_sensors + rolling_means + rolling_stds +
                rolling_mins + rolling_maxs + error_feats +
                time_feats + maint_feats + meta_feats)
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
print(f'Features : {len(FEATURE_COLS)}')
print(f'Shape    : {df.shape}')

In [ ]:
# Temporal train/test split — never random split for time series
cutoff   = pd.Timestamp('2015-09-01')
train_df = df[df['datetime'] <  cutoff].copy()
test_df  = df[df['datetime'] >= cutoff].copy()

X_train_raw = train_df[FEATURE_COLS].fillna(0).values
y_train     = train_df['failure_label'].values
X_test_raw  = test_df[FEATURE_COLS].fillna(0).values
y_test      = test_df['failure_label'].values

print(f'Train : {X_train_raw.shape}  positives: {y_train.sum():,} ({100*y_train.mean():.1f}%)')
print(f'Test  : {X_test_raw.shape}  positives: {y_test.sum():,} ({100*y_test.mean():.1f}%)')

In [ ]:
# Scale and SMOTE
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

print(f'Scaler fit on {scaler.n_features_in_} features')
print('Applying SMOTE (sampling_strategy=0.1) ...')

smote = SMOTE(sampling_strategy=0.1, random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'After SMOTE — neg: {(y_train_sm==0).sum():,}  pos: {(y_train_sm==1).sum():,}')
print(f'Ratio: {(y_train_sm==0).sum()/(y_train_sm==1).sum():.1f}:1')

In [ ]:
# Model 1 — Logistic Regression baseline
mlflow.set_experiment('failure_prediction')

with mlflow.start_run(run_name='logistic_regression'):
    params = {'C':1.0,'max_iter':500,'random_state':SEED,'n_jobs':-1}
    mlflow.log_params(params)

    lr_model = LogisticRegression(**params)
    lr_model.fit(X_train_sm, y_train_sm)

    lr_probs = lr_model.predict_proba(X_test)[:,1]
    lr_preds = (lr_probs >= 0.5).astype(int)
    f1_lr    = f1_score(y_test, lr_preds, zero_division=0)
    auc_lr   = roc_auc_score(y_test, lr_probs)

    mlflow.log_metric('f1_score', f1_lr)
    mlflow.log_metric('roc_auc',  auc_lr)
    mlflow.sklearn.log_model(lr_model, 'logistic_regression')

print('=== LOGISTIC REGRESSION ===')
print(f'F1 : {f1_lr:.4f}   ROC-AUC : {auc_lr:.4f}')
print(classification_report(y_test, lr_preds, target_names=['normal','failure']))

In [ ]:
# Model 2 — XGBoost
scale_pos_weight = (y_train_sm==0).sum() / (y_train_sm==1).sum()

with mlflow.start_run(run_name='xgboost_v1'):
    params = {
        'n_estimators':300,'max_depth':6,'learning_rate':0.05,
        'subsample':0.8,'colsample_bytree':0.8,
        'scale_pos_weight':scale_pos_weight,
        'eval_metric':'logloss','random_state':SEED,
        'n_jobs':-1,'tree_method':'hist',
    }
    mlflow.log_params(params)

    print('Training XGBoost...')
    xgb_model = xgb.XGBClassifier(**params)
    xgb_model.fit(X_train_sm, y_train_sm,
                  eval_set=[(X_test, y_test)], verbose=50)

    xgb_probs = xgb_model.predict_proba(X_test)[:,1]

    # Optimal threshold via PR curve
    p, r, t  = precision_recall_curve(y_test, xgb_probs)
    f1c      = 2*p*r/(p+r+1e-8)
    xgb_thr  = float(t[np.argmax(f1c)])
    xgb_preds = (xgb_probs >= xgb_thr).astype(int)
    f1_xgb   = f1_score(y_test, xgb_preds, zero_division=0)
    auc_xgb  = roc_auc_score(y_test, xgb_probs)

    mlflow.log_metric('f1_score',  f1_xgb)
    mlflow.log_metric('roc_auc',   auc_xgb)
    mlflow.log_metric('threshold', xgb_thr)
    mlflow.xgboost.log_model(xgb_model, 'xgboost')

print(f'\n=== XGBOOST (threshold={xgb_thr:.3f}) ===')
print(f'F1 : {f1_xgb:.4f}   ROC-AUC : {auc_xgb:.4f}')
print(classification_report(y_test, xgb_preds, target_names=['normal','failure']))

In [ ]:
# SHAP — why did the model predict failure?
print('Computing SHAP values (2-4 min)...')
explainer   = shap.TreeExplainer(xgb_model)
sample_idx  = np.random.choice(len(X_test), size=2000, replace=False)
X_sample    = X_test[sample_idx]
y_sample    = y_test[sample_idx]
shap_values = explainer.shap_values(X_sample)
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# SHAP beeswarm — most important plot for README
plt.figure(figsize=(10,8))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS,
                  max_display=20, show=False)
plt.title('SHAP Feature Importance — XGBoost Failure Prediction', fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'shap_beeswarm.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved shap_beeswarm.png')

In [ ]:
# SHAP bar chart
plt.figure(figsize=(9,6))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS,
                  plot_type='bar', max_display=15, show=False)
plt.title('Top 15 Features by Mean |SHAP| Value', fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'shap_bar.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved shap_bar.png')

In [ ]:
# SHAP waterfall — single prediction explanation
xgb_preds_sample = (xgb_model.predict_proba(X_sample)[:,1] >= xgb_thr).astype(int)
tp_idx = np.where((xgb_preds_sample == 1) & (y_sample == 1))[0]

if len(tp_idx) > 0:
    idx = tp_idx[0]
    exp = shap.Explanation(values=shap_values[idx],
                           base_values=explainer.expected_value,
                           data=X_sample[idx],
                           feature_names=FEATURE_COLS)
    plt.figure(figsize=(10,6))
    shap.waterfall_plot(exp, max_display=15, show=False)
    plt.title('Why did the model predict failure?', fontweight='bold')
    plt.tight_layout()
    plt.savefig(DATA_PROCESSED / 'shap_waterfall.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved shap_waterfall.png')
else:
    print('No true positives in sample — waterfall skipped')

In [ ]:
# Model 3 — 1D-CNN
# Build sequences per machine
SEQ_LEN = 6

def build_sequences_per_machine(df_subset, feature_cols, seq_len, sc):
    all_seqs, all_labels = [], []
    for _, group in df_subset.groupby('machineID'):
        group   = group.sort_values('datetime')
        X_group = sc.transform(group[feature_cols].fillna(0))
        y_group = group['failure_label'].values
        for i in range(len(X_group) - seq_len + 1):
            all_seqs.append(X_group[i:i+seq_len])
            all_labels.append(y_group[i+seq_len-1])
    return np.array(all_seqs), np.array(all_labels)

print('Building sequences...')
X_seq_tr, y_seq_tr = build_sequences_per_machine(train_df, FEATURE_COLS, SEQ_LEN, scaler)
X_seq_te, y_seq_te = build_sequences_per_machine(test_df,  FEATURE_COLS, SEQ_LEN, scaler)
print(f'Train: {X_seq_tr.shape}  pos: {y_seq_tr.sum():,}')
print(f'Test : {X_seq_te.shape}  pos: {y_seq_te.sum():,}')

In [ ]:
# SMOTE on sequences
n_tr, sl, nf = X_seq_tr.shape
X_flat = X_seq_tr.reshape(n_tr, -1)
smote2 = SMOTE(sampling_strategy=0.1, random_state=SEED)
X_flat_sm, y_seq_sm = smote2.fit_resample(X_flat, y_seq_tr)
X_seq_sm = X_flat_sm.reshape(-1, sl, nf)
print(f'After SMOTE: {X_seq_sm.shape}  pos: {y_seq_sm.sum():,}')

In [ ]:
# 1D-CNN architecture
class CNN1D(nn.Module):
    def __init__(self, input_dim, seq_len):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(input_dim, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2))
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2))
        self.pool  = nn.AdaptiveAvgPool1d(1)
        self.head  = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 1))

    def forward(self, x):
        x = x.permute(0, 2, 1)           # (batch, features, seq)
        x = self.conv2(self.conv1(x))
        x = self.pool(x).squeeze(-1)      # (batch, 128)
        return self.head(x).squeeze(-1)   # (batch,)

cnn_model = CNN1D(input_dim=nf, seq_len=SEQ_LEN).to(device)
n_params  = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f'1D-CNN params: {n_params:,}')

In [ ]:
# Train 1D-CNN
CNN_EPOCHS = 10
CNN_BS     = 512
CNN_LR     = 1e-3

X_tr_t = torch.FloatTensor(X_seq_sm).to(device)
y_tr_t = torch.FloatTensor(y_seq_sm).to(device)
X_te_t = torch.FloatTensor(X_seq_te).to(device)

train_ld   = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=CNN_BS, shuffle=True)
pos_weight = torch.tensor([(y_seq_sm==0).sum()/(y_seq_sm==1).sum()]).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.Adam(cnn_model.parameters(), lr=CNN_LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CNN_LR, steps_per_epoch=len(train_ld), epochs=CNN_EPOCHS)

print(f'Training 1D-CNN for {CNN_EPOCHS} epochs...')
print('Epoch | Loss')
print('-'*20)

cnn_losses = []
with mlflow.start_run(run_name='cnn1d_v1'):
    mlflow.log_params({'model':'CNN1D','seq_len':SEQ_LEN,'epochs':CNN_EPOCHS,
                       'batch_size':CNN_BS,'lr':CNN_LR})

    cnn_model.train()
    for epoch in range(1, CNN_EPOCHS+1):
        epoch_loss = 0.0
        for Xb, yb in train_ld:
            optimizer.zero_grad()
            loss = criterion(cnn_model(Xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(cnn_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(train_ld)
        cnn_losses.append(avg)
        mlflow.log_metric('train_loss', avg, step=epoch)
        print(f'  {epoch:>3}  |  {avg:.6f}')

    # Evaluate
    cnn_model.eval()
    cnn_probs_list = []
    with torch.no_grad():
        for i in range(0, len(X_te_t), 1024):
            logits = cnn_model(X_te_t[i:i+1024])
            cnn_probs_list.append(torch.sigmoid(logits).cpu().numpy())
    cnn_probs = np.concatenate(cnn_probs_list)

    p, r, t  = precision_recall_curve(y_seq_te, cnn_probs)
    f1c      = 2*p*r/(p+r+1e-8)
    cnn_thr  = float(t[np.argmax(f1c)])
    cnn_preds = (cnn_probs >= cnn_thr).astype(int)
    f1_cnn   = f1_score(y_seq_te, cnn_preds, zero_division=0)
    auc_cnn  = roc_auc_score(y_seq_te, cnn_probs)

    mlflow.log_metric('f1_score',  f1_cnn)
    mlflow.log_metric('roc_auc',   auc_cnn)
    mlflow.log_metric('threshold', cnn_thr)
    mlflow.pytorch.log_model(cnn_model, 'cnn1d')

print(f'\n=== 1D-CNN (threshold={cnn_thr:.3f}) ===')
print(f'F1 : {f1_cnn:.4f}   ROC-AUC : {auc_cnn:.4f}')
print(classification_report(y_seq_te, cnn_preds, target_names=['normal','failure']))

In [ ]:
# Final comparison
print('=== FINAL MODEL COMPARISON ===')
print(f'{"Model":<28} {"F1":>8} {"ROC-AUC":>10}')
print('-'*48)
print(f'{"Logistic Regression":<28} {f1_lr:>8.4f} {auc_lr:>10.4f}')
print(f'{"XGBoost":<28} {f1_xgb:>8.4f} {auc_xgb:>10.4f}')
print(f'{"1D-CNN":<28} {f1_cnn:>8.4f} {auc_cnn:>10.4f}')

In [ ]:
# Precision-Recall comparison plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for probs, ytrue, label, color in [
    (xgb_probs, y_test,   'XGBoost', '#185FA5'),
    (lr_probs,  y_test,   'LogReg',  '#BA7517'),
    (cnn_probs, y_seq_te, '1D-CNN',  '#1D9E75'),
]:
    p, r, _ = precision_recall_curve(ytrue, probs)
    axes[0].plot(r, p, label=label, color=color, linewidth=2)
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curves', fontweight='bold')
axes[0].legend()

cm   = confusion_matrix(y_test, xgb_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['Normal','Failure'])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('XGBoost — Confusion Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved model_comparison.png')

In [ ]:
# Save all artefacts
joblib.dump(xgb_model, MODELS_DIR / 'xgboost_failure.joblib')
joblib.dump(scaler,    MODELS_DIR / 'scaler_failure.joblib')
torch.save({'model_state_dict':cnn_model.state_dict(),
            'input_dim':nf,'seq_len':SEQ_LEN}, MODELS_DIR / 'cnn1d_failure.pt')

meta = {'feature_cols':FEATURE_COLS,'xgb_threshold':xgb_thr,
        'cnn_threshold':cnn_thr,'seq_len':SEQ_LEN,
        'xgb_f1':float(f1_xgb),'xgb_roc_auc':float(auc_xgb),
        'cnn_f1':float(f1_cnn),'cnn_roc_auc':float(auc_cnn)}
with open(MODELS_DIR / 'model_meta.json','w') as f:
    json.dump(meta, f, indent=2)

print('Saved:')
print('  models/xgboost_failure.joblib')
print('  models/scaler_failure.joblib')
print('  models/cnn1d_failure.pt')
print('  models/model_meta.json')
print()
print('=== NOTEBOOK 04 COMPLETE ===')
print('Next: notebook 05 — MLflow dashboard + FastAPI + Docker')